# Experiments 41-44 — Quarters and Eighths vs. Individual Poets (Not Merged)

Different design from Experiments 37-40: instead of one aggregate
vector (35Vector/19Vector) facing many Quranic units, this keeps every
poet as their own separate entity, clustered together with the
quarters/eighths -- the same style of comparison as the original
Experiments 8/9/10.

- **41:** 240 quarters + 35 individual poets = 275 total
- **42:** 240 quarters + 19 individual poets = 259 total
- **43:** 480 eighths + 35 individual poets = 515 total
- **44:** 480 eighths + 19 individual poets = 499 total

Output is cluster-membership analysis (how many quarters/eighths land
in a poet-containing cluster, and which poets those are) plus UMAP
graphs -- matching how Experiments 8/9/10 were reported, not the
single-vector similarity-ranking style of Experiments 37-40.

**Reuses all your cached embeddings** — no new model computation
needed, should run fast.

**Before you start:** put `poems.db` in the same folder as this
notebook — the **real** one (several hundred KB or more), not an
empty auto-created stand-in. If unsure, check the file size before
running.

Run cells top to bottom, **Shift+Enter**.

In [ ]:
# CELL 1 -- Install packages
!pip -q install sentence-transformers torch scikit-learn umap-learn hdbscan pandas numpy matplotlib seaborn scipy requests

In [ ]:
# CELL 2 -- Configuration
import re, sqlite3, hashlib, json, warnings, pickle
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

warnings.filterwarnings("ignore")

DB_PATH = Path("poems.db")
CACHE_DIR = Path("quran_cache"); CACHE_DIR.mkdir(exist_ok=True)
EMBED_CACHE_DIR = Path("embed_cache"); EMBED_CACHE_DIR.mkdir(exist_ok=True)
FIGURES_DIR = Path("output/figures"); FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = Path("output/tables"); TABLES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = Path("output/reports"); REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Sanity check on poems.db BEFORE doing anything else, so a bad file
# fails immediately with a clear message instead of a confusing error
# several cells later.
if not DB_PATH.exists():
    raise FileNotFoundError(
        "poems.db not found in this folder. Put the real poems.db "
        "(same one used in every other notebook in this project) here."
    )
if DB_PATH.stat().st_size < 10_000:
    raise ValueError(
        f"poems.db is only {DB_PATH.stat().st_size} bytes -- this is almost "
        "certainly an empty stand-in file, not the real corpus database. "
        "Delete it and find the real poems.db (several hundred KB or more)."
    )

SBERT_MODEL_NAME = "akhooli/Arabic-SBERT-100K"
MAX_VERSES_PER_POEM = 20

UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_N_COMPONENTS_HIGH = 50
UMAP_METRIC = "cosine"
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 3

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
import torch
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    print("No GPU found -- will run on CPU.")

def split_verses(poem_text):
    if not poem_text or not isinstance(poem_text, str):
        return []
    verses = re.split(r'[\n\r]+|[.!\u061F?\u061B;]+', poem_text)
    return [v.strip() for v in verses if len(v.strip()) > 10]

def normalize_word(w):
    w = re.sub(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED\u0670]", "", w)
    w = re.sub(r"\u0640", "", w)
    w = re.sub(r"[\u0622\u0623\u0625\u0671]", "\u0627", w)
    return w.strip()

plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 11})
print(f"Config loaded. poems.db size: {DB_PATH.stat().st_size:,} bytes -- looks like a real file.")

In [ ]:
# CELL 3 -- Load the 260 poets from poems.db
conn = sqlite3.connect(str(DB_PATH))
df = pd.read_sql_query(
    "SELECT poet_name, poem_title, poem_text, poem_type, poem_meter, verses_count "
    "FROM poems WHERE poet_name IS NOT NULL AND poem_text IS NOT NULL "
    "AND LENGTH(poem_text) >= 50",
    conn,
)
conn.close()

df["poem_hash"] = df["poem_text"].apply(lambda t: hashlib.md5(t.strip().encode("utf-8")).hexdigest())
df = df.drop_duplicates(subset=["poem_hash"]).copy()

poems_by_poet = defaultdict(list)
for _, row in df.iterrows():
    poems_by_poet[row["poet_name"]].append(row["poem_text"])
poems_by_poet = dict(poems_by_poet)

poet_total_verses = {
    poet: sum(len(split_verses(p)) for p in poems)
    for poet, poems in poems_by_poet.items()
}

print(f"Poets loaded: {len(poems_by_poet)}")

In [ ]:
# CELL 4 -- Fetch Quran text WITH hizb-quarter metadata
cache_file = CACHE_DIR / "quran_ayat_with_hizb.json"

if cache_file.exists():
    print("Loading Quran (with hizb metadata) from local cache...")
    with open(cache_file, encoding="utf-8") as f:
        surahs_raw = json.load(f)
else:
    print("Fetching Quran from Al Quran Cloud API...")
    resp = requests.get("https://api.alquran.cloud/v1/quran/quran-uthmani", timeout=60)
    resp.raise_for_status()
    surahs_raw = resp.json()["data"]["surahs"]
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(surahs_raw, f, ensure_ascii=False)
    print("Fetched and cached.")

surah_names = {}
surah_poem_text = {}
ayah_records = []
for s in surahs_raw:
    snum = s["number"]
    surah_names[snum] = s["englishName"]
    ayat_texts = [a["text"] for a in s["ayahs"]]
    surah_poem_text[snum] = "\n".join(ayat_texts)
    for a in s["ayahs"]:
        hizb_number = ((a["hizbQuarter"] - 1) // 4) + 1
        ayah_records.append({
            "surah_number": snum, "ayah_number": a["numberInSurah"],
            "text": a["text"], "hizb_number": hizb_number,
            "hizb_quarter": a["hizbQuarter"],
        })

ayah_df = pd.DataFrame(ayah_records)
ayah_df["global_order"] = range(len(ayah_df))
print(f"Surahs: {ayah_df['surah_number'].nunique()} | Ayat: {len(ayah_df)} | "
      f"Hizb quarters found: {ayah_df['hizb_quarter'].nunique()} (should be 240)")

In [ ]:
# CELL 5 -- Load poet embeddings (cached), build the 35-poet and 19-poet subsets (kept individual, not merged)
from sentence_transformers import SentenceTransformer

poet_cache_file = EMBED_CACHE_DIR / "poet_embeddings.pkl"

print("Loading model:", SBERT_MODEL_NAME)
model = SentenceTransformer(SBERT_MODEL_NAME)
print("Loaded.")

def _encode(texts, batch_size=64):
    if not texts:
        return np.array([])
    return model.encode(texts, batch_size=batch_size, show_progress_bar=False,
                         normalize_embeddings=True, convert_to_numpy=True)

def embed_poem_verse_average(poems_by_author, max_verses=MAX_VERSES_PER_POEM, seed=RANDOM_SEED):
    out = {}
    rng = np.random.RandomState(seed)
    for author, poems in poems_by_author.items():
        poem_vectors = []
        for poem in poems:
            verses = split_verses(poem)
            if len(verses) > max_verses:
                idx = rng.choice(len(verses), max_verses, replace=False)
                verses = [verses[i] for i in sorted(idx)]
            if verses:
                poem_vectors.append(np.mean(_encode(verses), axis=0))
        if poem_vectors:
            out[author] = np.mean(poem_vectors, axis=0)
    return out

if poet_cache_file.exists():
    print("Loading cached poet embeddings...")
    with open(poet_cache_file, "rb") as f:
        poet_embeddings = pickle.load(f)
    print(f"Loaded {len(poet_embeddings)} cached poet embeddings.")
else:
    print("Embedding 260 poets (slow, one-time only)...")
    poet_embeddings = embed_poem_verse_average(poems_by_poet)
    with open(poet_cache_file, "wb") as f:
        pickle.dump(poet_embeddings, f)
    print(f"Done and cached. {len(poet_embeddings)} poets embedded.")

max_surah_verses = max(len(split_verses(t)) for t in surah_poem_text.values())
poets_35 = [p for p, v in poet_total_verses.items() if v > max_surah_verses]
poets_35_embeddings = {p: poet_embeddings[p] for p in poets_35 if p in poet_embeddings}
print(f"\n35-poet subset (kept individual): {len(poets_35_embeddings)} poets")

In [ ]:
# CELL 6 -- Load ayah embeddings (cached), build quarter/eighth/whole vectors
ayah_embed_cache_file = EMBED_CACHE_DIR / "ayah_embeddings.pkl"

if ayah_embed_cache_file.exists():
    print("Loading cached ayah embeddings...")
    with open(ayah_embed_cache_file, "rb") as f:
        ayah_embeddings = pickle.load(f)
    print(f"Loaded {len(ayah_embeddings)} cached ayah embeddings.")
else:
    print(f"Embedding all {len(ayah_df)} ayat individually (slow, one-time only)...")
    all_texts = ayah_df["text"].tolist()
    all_vecs = _encode(all_texts, batch_size=64)
    ayah_embeddings = {}
    for (snum, anum), vec in zip(zip(ayah_df["surah_number"], ayah_df["ayah_number"]), all_vecs):
        ayah_embeddings[(snum, anum)] = vec
    with open(ayah_embed_cache_file, "wb") as f:
        pickle.dump(ayah_embeddings, f)
    print(f"Done and cached. {len(ayah_embeddings)} ayat embedded.")

def vector_for_ayat(ayat_keys):
    vecs = [ayah_embeddings[k] for k in ayat_keys if k in ayah_embeddings]
    return np.mean(vecs, axis=0) if vecs else None

quarter_vectors = {}
for qnum, group in ayah_df.groupby("hizb_quarter"):
    keys = list(zip(group["surah_number"], group["ayah_number"]))
    v = vector_for_ayat(keys)
    if v is not None:
        quarter_vectors[qnum] = v
print(f"Quarter vectors built: {len(quarter_vectors)} (should be 240)")

eighth_vectors = {}
for qnum, group in ayah_df.groupby("hizb_quarter"):
    group_sorted = group.sort_values("global_order")
    n = len(group_sorted)
    half = (n + 1) // 2
    first_half = group_sorted.iloc[:half]
    second_half = group_sorted.iloc[half:]
    keys_a = list(zip(first_half["surah_number"], first_half["ayah_number"]))
    keys_b = list(zip(second_half["surah_number"], second_half["ayah_number"]))
    v_a = vector_for_ayat(keys_a)
    v_b = vector_for_ayat(keys_b)
    if v_a is not None:
        eighth_vectors[f"{qnum}a"] = v_a
    if v_b is not None:
        eighth_vectors[f"{qnum}b"] = v_b
print(f"Eighth vectors built: {len(eighth_vectors)} (should be 480)")

surah_vectors = {}
for snum in surah_names:
    keys = list(zip(ayah_df[ayah_df["surah_number"] == snum]["surah_number"],
                    ayah_df[ayah_df["surah_number"] == snum]["ayah_number"]))
    v = vector_for_ayat(keys)
    if v is not None:
        surah_vectors[snum] = v
quran_whole_vector = np.mean(list(surah_vectors.values()), axis=0)
print(f"Surah vectors (for identifying the 19 poets): {len(surah_vectors)}")

In [ ]:
# CELL 7 -- Clustering helper, then identify the 19-poet subset (rerun of Experiment 1's setup)
import umap, hdbscan
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score

def run_umap(matrix, n_components, n_neighbors, min_dist, metric="cosine", random_state=RANDOM_SEED):
    reducer = umap.UMAP(n_neighbors=min(n_neighbors, len(matrix) - 1), n_components=n_components,
                         min_dist=min_dist, metric=metric, random_state=random_state)
    return reducer.fit_transform(matrix)

def cluster_and_report(embeddings_dict, label_types, title, n_neighbors=UMAP_N_NEIGHBORS,
                       min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, verbose=True):
    names = sorted(embeddings_dict.keys())
    n = len(names)
    emb_matrix = np.array([embeddings_dict[nm] for nm in names])
    umap_high = run_umap(emb_matrix, min(UMAP_N_COMPONENTS_HIGH, max(2, n - 2)), n_neighbors, 0.0, UMAP_METRIC)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min(min_cluster_size, max(2, n // 10)),
                                min_samples=HDBSCAN_MIN_SAMPLES,
                                cluster_selection_method="eom", metric="euclidean")
    labels = clusterer.fit_predict(umap_high)
    umap_2d = run_umap(emb_matrix, 2, n_neighbors, UMAP_MIN_DIST, UMAP_METRIC)
    mask = labels >= 0
    n_clusters = len(set(labels[mask])) if mask.sum() > 0 else 0
    n_outliers = int(np.sum(labels == -1))
    sil = float(silhouette_score(umap_high[mask], labels[mask])) if n_clusters >= 2 and mask.sum() > n_clusters else None
    result_df = pd.DataFrame({"name": names, "type": [label_types[nm] for nm in names],
        "cluster": labels, "umap_x": umap_2d[:, 0], "umap_y": umap_2d[:, 1]})
    if verbose:
        print(f"=== {title} ===")
        print(f"Total: {n} | Clusters: {n_clusters} | Outliers: {n_outliers} | Silhouette: {sil}")
    return result_df, {"n_entities": n, "n_clusters": n_clusters, "n_outliers": n_outliers, "silhouette": sil}

print("Identifying the 19-poet subset by rerunning Experiment 1 (all 260 poets + Quran whole)...")
exp1_rerun_embeddings = dict(poet_embeddings)
exp1_rerun_embeddings["Quran (whole)"] = quran_whole_vector
names_1 = sorted(exp1_rerun_embeddings.keys())
n_1 = len(names_1)
emb_matrix_1 = np.array([exp1_rerun_embeddings[nm] for nm in names_1])
umap_high_1 = run_umap(emb_matrix_1, min(UMAP_N_COMPONENTS_HIGH, n_1 - 2), UMAP_N_NEIGHBORS, 0.0)
clusterer_1 = hdbscan.HDBSCAN(min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, min_samples=HDBSCAN_MIN_SAMPLES,
                              cluster_selection_method="eom", metric="euclidean")
labels_1 = clusterer_1.fit_predict(umap_high_1)
labels_1_by_name = dict(zip(names_1, labels_1))
quran_cluster_1 = labels_1_by_name["Quran (whole)"]
poets_19_names = [nm for nm in names_1 if nm != "Quran (whole)" and labels_1_by_name[nm] == quran_cluster_1]
poets_19_embeddings = {p: poet_embeddings[p] for p in poets_19_names}
print(f"19-poet subset identified: {len(poets_19_embeddings)} poets sharing the Quran's cluster in this rerun")
print("(small variation from the original 19 is normal run-to-run UMAP/HDBSCAN noise).")

## Experiments 41-42 — 240 Quarters vs. Individual Poets (35, then 19)

In [ ]:
# CELL 8 -- Experiments 41 and 42: 240 quarters vs individual poets (35, then 19)
def cluster_many_vs_many(poet_dict, poet_label_count, units_dict, unit_label, exp_num, title_note):
    embeddings = {**poet_dict, **units_dict}
    types = {nm: ("Poet" if nm in poet_dict else unit_label) for nm in embeddings}
    result_df, metrics = cluster_and_report(embeddings, types,
        f"Experiment {exp_num}: {poet_label_count} poets + {unit_label} ({title_note})")

    comp = result_df.groupby("cluster")["type"].value_counts().unstack(fill_value=0)
    # Exclude the noise bucket (cluster == -1) from "mixed cluster" counting --
    # noise isn't a real cluster, and leaving it in would double-count any
    # quarter/eighth that happens to be classified as noise alongside a poet
    # that's also noise, producing an impossible negative remainder.
    comp_real_clusters = comp[comp.index != -1]
    mixed = comp_real_clusters[(comp_real_clusters.get("Poet", 0) > 0) & (comp_real_clusters.get(unit_label, 0) > 0)]
    unit_rows = result_df[result_df["type"] == unit_label]
    units_in_mixed = sum(comp_real_clusters.loc[mixed.index, unit_label]) if len(mixed) > 0 else 0
    unit_outliers = unit_rows[unit_rows["cluster"] == -1]

    print(f"\n{unit_label}s in mixed clusters (sharing with >=1 poet): "
          f"{units_in_mixed} / {len(unit_rows)} ({units_in_mixed/len(unit_rows)*100:.1f}%)")
    print(f"{unit_label}s classified as noise: {len(unit_outliers)} / {len(unit_rows)}")

    if len(mixed) > 0:
        print(f"\nMixed clusters -- which poets share a cluster with {unit_label.lower()}s:")
        for cl in mixed.index:
            members = result_df[result_df["cluster"] == cl]
            poets_in = members[members["type"] == "Poet"]["name"].tolist()
            n_units_in = len(members[members["type"] == unit_label])
            print(f"  Cluster {cl}: {n_units_in} {unit_label.lower()}s with poets: {poets_in}")

    return result_df, metrics, {"units_in_mixed": int(units_in_mixed), "n_units": len(unit_rows),
                                "n_unit_outliers": len(unit_outliers), "n_mixed_clusters": len(mixed)}

quarter_units = {f"Quarter {n}": v for n, v in quarter_vectors.items()}

# --- Experiment 41: 240 quarters + 35 individual poets ---
exp41_df, exp41_metrics, exp41_summary = cluster_many_vs_many(
    poets_35_embeddings, 35, quarter_units, "Quarter", 41, "N=275")
exp41_df.to_csv(TABLES_DIR / "experiment41_clusters.csv", index=False)

# --- Experiment 42: 240 quarters + 19 individual poets ---
print("\n" + "=" * 70)
exp42_df, exp42_metrics, exp42_summary = cluster_many_vs_many(
    poets_19_embeddings, 19, quarter_units, "Quarter", 42, "N=259")
exp42_df.to_csv(TABLES_DIR / "experiment42_clusters.csv", index=False)

## Experiments 43-44 — 480 Eighths vs. Individual Poets (35, then 19)

In [ ]:
# CELL 9 -- Experiments 43 and 44: 480 eighths vs individual poets (35, then 19)
eighth_units = {f"Eighth {n}": v for n, v in eighth_vectors.items()}

# --- Experiment 43: 480 eighths + 35 individual poets ---
exp43_df, exp43_metrics, exp43_summary = cluster_many_vs_many(
    poets_35_embeddings, 35, eighth_units, "Eighth", 43, "N=515")
exp43_df.to_csv(TABLES_DIR / "experiment43_clusters.csv", index=False)

# --- Experiment 44: 480 eighths + 19 individual poets ---
print("\n" + "=" * 70)
exp44_df, exp44_metrics, exp44_summary = cluster_many_vs_many(
    poets_19_embeddings, 19, eighth_units, "Eighth", 44, "N=499")
exp44_df.to_csv(TABLES_DIR / "experiment44_clusters.csv", index=False)

In [ ]:
# CELL 10 -- Figures
def plot_many_vs_many(df, title, save_name, unit_marker):
    fig, ax = plt.subplots(figsize=(12, 9))
    unique_clusters = sorted(df["cluster"].unique())
    n_clust_plot = len([c for c in unique_clusters if c >= 0])
    colors = plt.cm.tab20(np.linspace(0, 1, max(n_clust_plot, 1)))
    for cl in unique_clusters:
        sub = df[df["cluster"] == cl]
        color = "gray" if cl == -1 else colors[cl % len(colors)]
        poets = sub[sub["type"] == "Poet"]
        units = sub[sub["type"] != "Poet"]
        if len(poets) > 0:
            ax.scatter(poets["umap_x"], poets["umap_y"], c=[color], marker="o", s=70,
                      alpha=0.85, edgecolors="black", linewidth=0.4)
        if len(units) > 0:
            ax.scatter(units["umap_x"], units["umap_y"], c=[color], marker=unit_marker, s=50,
                      alpha=0.7, edgecolors="black", linewidth=0.3)
    ax.set_xlabel("UMAP Dimension 1"); ax.set_ylabel("UMAP Dimension 2")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / save_name, dpi=300, bbox_inches="tight")
    plt.show()

plot_many_vs_many(exp41_df, "Experiment 41: 35 Poets + 240 Quarters (N=275)", "experiment41_umap.png", "^")
plot_many_vs_many(exp42_df, "Experiment 42: 19 Poets + 240 Quarters (N=259)", "experiment42_umap.png", "^")
plot_many_vs_many(exp43_df, "Experiment 43: 35 Poets + 480 Eighths (N=515)", "experiment43_umap.png", "v")
plot_many_vs_many(exp44_df, "Experiment 44: 19 Poets + 480 Eighths (N=499)", "experiment44_umap.png", "v")

print("All figures saved.")

In [ ]:
# CELL 11 -- Final report
report_path = REPORTS_DIR / "experiments_41_44_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("=" * 70 + "\n")
    f.write("EXPERIMENTS 41-44: QUARTERS AND EIGHTHS VS INDIVIDUAL POETS\n")
    f.write("=" * 70 + "\n\n")

    for exp_num, metrics, summary, label in [
        (41, exp41_metrics, exp41_summary, "35 poets + 240 quarters"),
        (42, exp42_metrics, exp42_summary, "19 poets + 240 quarters"),
        (43, exp43_metrics, exp43_summary, "35 poets + 480 eighths"),
        (44, exp44_metrics, exp44_summary, "19 poets + 480 eighths"),
    ]:
        f.write(f"EXPERIMENT {exp_num}: {label}\n" + "-" * 40 + "\n")
        for k, v in metrics.items():
            f.write(f"  {k}: {v}\n")
        f.write(f"  Units in mixed clusters: {summary['units_in_mixed']} / {summary['n_units']} "
                f"({summary['units_in_mixed']/summary['n_units']*100:.1f}%)\n")
        f.write(f"  Units classified as noise: {summary['n_unit_outliers']} / {summary['n_units']}\n")
        f.write(f"  Mixed clusters: {summary['n_mixed_clusters']}\n\n")

print(f"Report written to {report_path}")
print()
print(open(report_path, encoding="utf-8").read())

## Done

Output in `output/`:
- `output/tables/experiment41/42/43/44_clusters.csv` — full cluster
  membership for every poet and every quarter/eighth
- `output/figures/` — UMAP plots for all four
- `output/reports/experiments_41_44_report.txt` — everything together

Send this back and I'll write the short report.